# Feature Engineering

This notebook performs following tasks-
- Raw data is loaded from an Athena table.
- Feature engineering techniques are applied—including binary label encoding, feature selection , and nan removal.
-  Random Forest classifier is used to select top 20 features.
-  Final engineered dataset is saved as a CSV and uploaded to an S3 bucket.
-  Created feature store with selected 20 features.
-  A new Athena table is registered to enable querying feature engineered dataset from feature store.


## Setup

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import boto3
from pyathena import connect
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import sys
import os
import io

In [2]:
# Read the config
sys.path.append('../config')
import config

In [3]:
# Read paths from config
bucket = config.S3_BUCKET
s3_staging_prefix = config.ATHENA_STAGING_PREFIX
s3_staging_path = f's3://{bucket}/{s3_staging_prefix}/'
db = config.ATHENA_DB_NAME
table = config.DATA_TABLE_NAME

# Set the feature store path
s3_engineered_path = f's3://{bucket}/final_project/feature_engineer/'
engineered_filename = 'engineered_features.csv'

In [4]:
# Setup AWS session
session = boto3.session.Session()
region = session.region_name

# start s3 client
s3_client = session.client('s3', region_name=region)

In [5]:
# List files in the prefix
prefix = 'final_project/staging/'
response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)

# Display
if 'Contents' in response:
    print(f"Objects under s3://{bucket}/{s3_staging_path}:")
    for obj in response['Contents']:
        print(" -", obj['Key'])
else:
    print("No objects found.")

Objects under s3://sagemaker-us-east-1-249645693565/s3://sagemaker-us-east-1-249645693565/s3://sagemaker-us-east-1-249645693565/final_project/staging//:
 - final_project/staging/19802747-90fb-4088-8a16-38af7135a303.txt
 - final_project/staging/19802747-90fb-4088-8a16-38af7135a303.txt.metadata
 - final_project/staging/98b15af1-b9d4-4518-a2f6-5515a6389c40.txt
 - final_project/staging/a073fa66-e4dd-42ac-82fd-7e52c2462ae2.txt
 - final_project/staging/c38f2899-5e2f-4f6a-b9b9-b717377a7b56.csv
 - final_project/staging/c38f2899-5e2f-4f6a-b9b9-b717377a7b56.csv.metadata


# Feature engineering

In [7]:
# Connect to staging directory and read table

conn = connect(region_name=region, s3_staging_dir=s3_staging_path)
query = f"SELECT * FROM {db}.{table}"
df = pd.read_sql(query, conn)

# shuffel the data
df = df.sample(frac=1, random_state=42).reset_index(drop=True)


/tmp/ipykernel_729/1520695368.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [8]:
df['label'].value_counts()

label
DDoS      128014
Benign     93250
Name: count, dtype: int64

In [10]:
df.head()

,protocol,flow_duration,total_fwd_packets,total_backward_packets,fwd_packets_length_total,bwd_packets_length_total,fwd_packet_length_max,fwd_packet_length_min,fwd_packet_length_mean,fwd_packet_length_std,...,fwd_seg_size_min,active_mean,active_std,active_max,active_min,idle_mean,idle_std,idle_max,idle_min,label
0,6,77017392,8,4,56,11601,20,0,7.000000,5.656854,...,20,3770298.0,0.00,3770298,3770298,73200000.0,0.00,73200000,73200000,DDoS
1,6,20726723,18,15,1343,20843,708,0,74.611115,199.540700,...,32,74318.5,80964.43,131569,17068,9985476.0,36024.97,10000000,9960002,Benign
2,6,3567947,4,0,24,0,6,6,6.000000,0.000000,...,20,0.0,0.00,0,0,0.0,0.00,0,0,DDoS
3,6,81260687,8,5,56,11607,20,0,7.000000,5.656854,...,20,1023.0,0.00,1023,1023,40100000.0,48200000.00,74100000,6048854,DDoS
4,17,252,2,2,64,258,32,32,32.000000,0.000000,...,32,0.0,0.00,0,0,0.0,0.00,0,0,Benign


In [11]:
nan_counts = df.isna().sum()
print(nan_counts[nan_counts > 1])

flow_bytes_per_s        221264
flow_packets_per_s      221264
fwd_packets_per_s       221264
bwd_packets_per_s       221264
down_up_ratio           221264
fwd_avg_bytes_bulk      221264
fwd_avg_packets_bulk    221264
bwd_avg_bytes_bulk      221264
bwd_avg_packets_bulk    221264
dtype: int64


In [12]:
# Drop columns with all NaNs
df_cleaned = df.dropna(axis=1, thresh=1)

In [13]:
df_cleaned.shape

(221264, 69)

In [14]:
df_cleaned['label'].unique()

array(['DDoS', 'Benign'], dtype=object)

In [15]:
# Binary label encoding
df_cleaned['label'] = df_cleaned['label'].apply(lambda x: 0 if str(x).lower() == 'benign' else 1)

/tmp/ipykernel_729/2488267272.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned['label'] = df_cleaned['label'].apply(lambda x: 0 if str(x).lower() == 'benign' else 1)


In [16]:
print(df_cleaned.shape, df_cleaned.dtypes)

(221264, 69) protocol                      int64
flow_duration                 int64
total_fwd_packets             int64
total_backward_packets        int64
fwd_packets_length_total      int64
                             ...   
idle_mean                   float64
idle_std                    float64
idle_max                      int64
idle_min                      int64
label                         int64
Length: 69, dtype: object


In [17]:
# Normalize numeric features
features = df.drop('label', axis=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)
X_scaled_df = pd.DataFrame(X_scaled, columns=features.columns)
X_scaled_df['label'] = df['label']

/opt/conda/lib/python3.12/site-packages/sklearn/utils/extmath.py:1137: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/opt/conda/lib/python3.12/site-packages/sklearn/utils/extmath.py:1142: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/opt/conda/lib/python3.12/site-packages/sklearn/utils/extmath.py:1162: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [18]:
# Feature selection using Random Forest
X = X_scaled_df.drop('label', axis=1)
y = X_scaled_df['label']
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X, y)

RandomForestClassifier(random_state=42)

In [19]:
# Select top 10 features
importances = clf.feature_importances_
top_features = X.columns[np.argsort(importances)[::-1][:10]]


In [20]:
# df_final dataframe with final selected features will be stored. df_final can be used by the model to train/validate and test.
df_final = df_cleaned[top_features.tolist() + ['label']]

In [21]:
print(top_features)

Index(['subflow_fwd_bytes', 'fwd_packet_length_mean',
       'fwd_packets_length_total', 'avg_fwd_segment_size',
       'fwd_act_data_packets', 'fwd_packet_length_max', 'fwd_iat_std',
       'init_fwd_win_bytes', 'fwd_header_length', 'subflow_fwd_packets'],
      dtype='object')


## Ingest Data into FeatureStore

### Feature store setup

In [36]:
from sagemaker.session import Session
sagemaker_client = session.client(service_name="sagemaker", region_name=region)
featurestore_runtime = session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

feature_store_session = Session(
    boto_session=session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

### Define FeatureGroups

In [37]:
import time
from time import gmtime, strftime, sleep

intrusion_feature_group_name = "feature-group-" + strftime("%d-%H-%M-%S", gmtime())

In [38]:
intrusion_feature_group_name

'feature-group-08-20-42-41'

In [39]:
from sagemaker.feature_store.feature_group import FeatureGroup

intrusion_feature_group = FeatureGroup(
    name=intrusion_feature_group_name, sagemaker_session=feature_store_session
)


In [40]:
import uuid
current_time_sec = int(round(time.time()))

# Move target column to front
target_col = "label"
df_final = df_final[[target_col] + [col for col in df_final.columns if col != target_col]]

# Add required metadata columns
record_identifier_feature_name = "record_id"
df_final[record_identifier_feature_name] = [str(uuid.uuid4()) for _ in range(len(df_final))]
event_time_feature_name = 'event_time'
df_final[event_time_feature_name] = pd.Series([current_time_sec] * len(df_final), dtype="float64")

In [41]:
# load feature definitions to the feature group. SageMaker FeatureStore Python SDK will auto-detect the data schema based on input data.
intrusion_feature_group.load_feature_definitions(data_frame=df_final)

[FeatureDefinition(feature_name='label', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='subflow_fwd_bytes', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='fwd_packet_length_mean', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='fwd_packets_length_total', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='avg_fwd_segment_size', feature_type=<FeatureTypeEnum.FRACTIONAL: 'Fractional'>, collection_type=None),
 FeatureDefinition(feature_name='fwd_act_data_packets', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='fwd_packet_length_max', feature_type=<FeatureTypeEnum.INTEGRAL: 'Integral'>, collection_type=None),
 FeatureDefinition(feature_name='fwd_iat_std', feature_type=<FeatureTypeEnum.FR

### Create FeatureGroups in SageMaker FeatureStore

In [42]:
from sagemaker import get_execution_role

# You can modify the following to use a role of your choosing. See the documentation for how to create this.
role = get_execution_role()
print(role)

arn:aws:iam::249645693565:role/LabRole


In [43]:
def wait_for_feature_group_creation_complete(feature_group):
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for Feature Group Creation")
        time.sleep(5)
        status = feature_group.describe().get("FeatureGroupStatus")
    if status != "Created":
        raise RuntimeError(f"Failed to create feature group {feature_group.name}")
    print(f"FeatureGroup {feature_group.name} successfully created.")


intrusion_feature_group.create(
    s3_uri= s3_engineered_path,
    record_identifier_name= record_identifier_feature_name,
    event_time_feature_name= event_time_feature_name,
    role_arn=role,
    enable_online_store=True,    
)


wait_for_feature_group_creation_complete(feature_group=intrusion_feature_group)

Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
Waiting for Feature Group Creation
FeatureGroup feature-group-08-20-42-41 successfully created.


In [44]:
s3_engineered_path

's3://sagemaker-us-east-1-249645693565/final_project/feature_engineer/'

In [45]:
# Confirm the FeatureGroup has been created by using the DescribeFeatureGroup and ListFeatureGroups APIs.
intrusion_feature_group.describe()

{'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:249645693565:feature-group/feature-group-08-20-42-41',
 'FeatureGroupName': 'feature-group-08-20-42-41',
 'RecordIdentifierFeatureName': 'record_id',
 'EventTimeFeatureName': 'event_time',
 'FeatureDefinitions': [{'FeatureName': 'label', 'FeatureType': 'Integral'},
  {'FeatureName': 'subflow_fwd_bytes', 'FeatureType': 'Integral'},
  {'FeatureName': 'fwd_packet_length_mean', 'FeatureType': 'Fractional'},
  {'FeatureName': 'fwd_packets_length_total', 'FeatureType': 'Integral'},
  {'FeatureName': 'avg_fwd_segment_size', 'FeatureType': 'Fractional'},
  {'FeatureName': 'fwd_act_data_packets', 'FeatureType': 'Integral'},
  {'FeatureName': 'fwd_packet_length_max', 'FeatureType': 'Integral'},
  {'FeatureName': 'fwd_iat_std', 'FeatureType': 'Fractional'},
  {'FeatureName': 'init_fwd_win_bytes', 'FeatureType': 'Integral'},
  {'FeatureName': 'fwd_header_length', 'FeatureType': 'Integral'},
  {'FeatureName': 'subflow_fwd_packets', 'FeatureType': 'In

In [46]:
#sagemaker_client.list_feature_groups() 

In [47]:
prefix_fs = 'final_project/engineered/'

In [48]:
# Convert DataFrame to CSV in memory
csv_buffer = io.StringIO()
df_final.to_csv(csv_buffer, index=False)

# Upload to S3
s3_client.put_object(
    Bucket=bucket,
    Key=f'{prefix_fs}/{engineered_filename}',
    Body=csv_buffer.getvalue()
)

{'ResponseMetadata': {'RequestId': 'K87EFCW07HM42VK3',
  'HostId': 'Ly0jhSR5eSNX1ZXXZKgtU4u9IczU9yZAgA+dBZGQ/pdkeZllJRM/gQphyFsZ8rZgBqbIyHgN6i7STTomzT25CdzHXnDritmXGK6kA0lPhtI=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'Ly0jhSR5eSNX1ZXXZKgtU4u9IczU9yZAgA+dBZGQ/pdkeZllJRM/gQphyFsZ8rZgBqbIyHgN6i7STTomzT25CdzHXnDritmXGK6kA0lPhtI=',
   'x-amz-request-id': 'K87EFCW07HM42VK3',
   'date': 'Sun, 08 Jun 2025 20:51:27 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"c60f1fb46aa03a95f191b3a1f2ecb3d9"',
   'x-amz-checksum-crc32': 'Pb2e/Q==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"c60f1fb46aa03a95f191b3a1f2ecb3d9"',
 'ChecksumCRC32': 'Pb2e/Q==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}

### Put records into FeatureGroup

In [49]:
intrusion_feature_group.ingest(data_frame=df_final, max_workers=3, wait=True)

IngestionManagerPandas(feature_group_name='feature-group-08-20-42-41', feature_definitions={'label': {'FeatureName': 'label', 'FeatureType': 'Integral'}, 'subflow_fwd_bytes': {'FeatureName': 'subflow_fwd_bytes', 'FeatureType': 'Integral'}, 'fwd_packet_length_mean': {'FeatureName': 'fwd_packet_length_mean', 'FeatureType': 'Fractional'}, 'fwd_packets_length_total': {'FeatureName': 'fwd_packets_length_total', 'FeatureType': 'Integral'}, 'avg_fwd_segment_size': {'FeatureName': 'avg_fwd_segment_size', 'FeatureType': 'Fractional'}, 'fwd_act_data_packets': {'FeatureName': 'fwd_act_data_packets', 'FeatureType': 'Integral'}, 'fwd_packet_length_max': {'FeatureName': 'fwd_packet_length_max', 'FeatureType': 'Integral'}, 'fwd_iat_std': {'FeatureName': 'fwd_iat_std', 'FeatureType': 'Fractional'}, 'init_fwd_win_bytes': {'FeatureName': 'init_fwd_win_bytes', 'FeatureType': 'Integral'}, 'fwd_header_length': {'FeatureName': 'fwd_header_length', 'FeatureType': 'Integral'}, 'subflow_fwd_packets': {'Feature

In [50]:
# Get offline store S3 URI
response = sagemaker_client.describe_feature_group(FeatureGroupName=intrusion_feature_group_name)
offline_store_uri = response["OfflineStoreConfig"]["S3StorageConfig"]["S3Uri"]
print("Offline store S3 location:", offline_store_uri)

Offline store S3 location: s3://sagemaker-us-east-1-249645693565/final_project/feature_engineer/


In [51]:
response = s3_client.list_objects_v2(Bucket=bucket, Prefix='final_project/engineered')

print("Files under offline store path:")
for obj in response.get("Contents", []):
    print(" -", obj['Key'])    

Files under offline store path:
 - final_project/engineered//engineered_features.csv


In [52]:
intrusion_feature_group_name

'feature-group-08-20-42-41'

## Query using Athena

In [53]:
from pyathena import connect

# Athena setup
athena_table = intrusion_feature_group_name.replace("-", "_") 

S3_FEATURE_STAGING_DIR = f"s3://{bucket}/final_project/staging_feature"

# Query data
conn = connect(region_name=region, s3_staging_dir=S3_FEATURE_STAGING_DIR)

In [54]:
# List all databases
db_list = pd.read_sql("SHOW DATABASES", conn)
display(db_list)

/tmp/ipykernel_729/2600548693.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  db_list = pd.read_sql("SHOW DATABASES", conn)


,database_name
0,default
1,dsoaws
2,final_project
3,final_project_featurestore
4,sagemaker_featurestore


In [55]:
# Run SHOW TABLES query
df_tables = pd.read_sql("SHOW TABLES IN sagemaker_featurestore", conn)
print(df_tables['tab_name'].tolist())

/tmp/ipykernel_729/2332857746.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tables = pd.read_sql("SHOW TABLES IN sagemaker_featurestore", conn)


['feature_group_05_22_36_29_1749163198', 'feature_group_08_20_27_30_1749414470', 'feature_group_08_20_42_41_1749415836', 'feature_group_1748487672', 'feature_group_29_03_25_36_1748489152', 'feature_group_29_16_11_06_1748535102', 'feature_group_29_17_07_31_1748538976', 'feature_group_29_17_41_57_1748540518', 'intrusion_feature_group_29_00_44_36_1748479608']


In [56]:
response = sagemaker_client.describe_feature_group(FeatureGroupName=intrusion_feature_group_name)
table_name = response["OfflineStoreConfig"]["DataCatalogConfig"]["TableName"]
print("Athena table name:", table_name)

Athena table name: feature_group_08_20_42_41_1749415836


In [57]:
athena_database = "sagemaker_featurestore" # Set by sagemaker
query = f'SELECT * FROM "{athena_database}"."{table_name}"'
df1 = pd.read_sql(query, conn)
df1.head()

/tmp/ipykernel_729/3676322808.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df1 = pd.read_sql(query, conn)


,label,subflow_fwd_bytes,fwd_packet_length_mean,fwd_packets_length_total,avg_fwd_segment_size,fwd_act_data_packets,fwd_packet_length_max,fwd_iat_std,init_fwd_win_bytes,fwd_header_length,subflow_fwd_packets,record_id,event_time,write_time,api_invocation_time,is_deleted
0,1,24,6.0,24,6.0,3,6,5740364.00,256,80,4,557d95f2-6951-4835-942a-d553d410bd71,1.749416e+09,2025-06-08 21:01:27.307,2025-06-08 20:56:26,False
1,0,64,32.0,64,32.0,1,32,0.00,-1,80,2,27cc1fd2-be13-4676-8ba0-74c0513b4c51,1.749416e+09,2025-06-08 20:56:27.281,2025-06-08 20:51:33,False
2,1,30,6.0,30,6.0,4,6,5626377.00,256,100,5,2216bb39-e77a-4aa0-b262-dd2cc08bf304,1.749416e+09,2025-06-08 20:56:27.502,2025-06-08 20:51:32,False
3,0,88,44.0,88,44.0,1,44,0.00,-1,40,2,dfb6163f-80dc-4c0c-820c-2e286651b4a9,1.749416e+09,2025-06-08 20:56:27.281,2025-06-08 20:51:33,False
4,1,30,6.0,30,6.0,4,6,189829.73,256,100,5,a9d4158e-ad0a-4323-9d10-1f7d8632781d,1.749416e+09,2025-06-08 20:56:27.281,2025-06-08 20:51:33,False


In [58]:
df1.shape

(156258, 16)

## Store paths in config

In [59]:
# Append feature output path to config.py
with open('../config/config.py', 'a') as f:
    f.write(f"FEATURE_ENGINEERED_CSV_PATH = 's3://{bucket}/final_project/engineered/{engineered_filename}'\n") 
    f.write(f"FEATURESTORE_DB = '{athena_database}'\n")
    f.write(f"FEATURESTORE_TABLE = '{table_name}'\n")
    f.write(f"S3_FEATURE_STAGING_DIR = '{S3_FEATURE_STAGING_DIR}'\n")

## Clean up

In [60]:
# Remove the Feature Groups
intrusion_feature_group.delete()

In [61]:
print(f"Feature-engineering completed.")

Feature-engineering completed.
